# budjet

Load Dataset

In [1]:
import pandas as pd
import numpy as np

df6 = pd.read_csv("../Datasets/budjet.csv")
print("[INFO] Loaded:", df6.shape)
df6.head()

[INFO] Loaded: (4436, 3)


,date,category,amount
0,2022-07-06 05:57:10 +0000,Restuarant,5.50
1,2022-07-06 05:57:27 +0000,Market,2.00
2,2022-07-06 05:58:12 +0000,Coffe,30.10
3,2022-07-06 05:58:25 +0000,Market,17.33
4,2022-07-06 05:59:00 +0000,Restuarant,5.50


Check Missing Values & Types

In [2]:
df6.info()
print(df6.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4436 entries, 0 to 4435
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   date      4436 non-null   object 
 1   category  4436 non-null   object 
 2   amount    4436 non-null   float64
dtypes: float64(1), object(2)
memory usage: 104.1+ KB
date        0
category    0
amount      0
dtype: int64


Check Duplications

In [3]:
df6.duplicated().sum() 

np.int64(0)

In [4]:
# let's see first few of duplicates
df6[df6.duplicated(keep=False)].sort_values(by=list(df6.columns)).head(10)

,date,category,amount


In [5]:
# let's remove one duplicating row 
df6 = df6.drop_duplicates()

In [6]:
# check again
print("After removing duplicates:", df6.shape)

After removing duplicates: (4436, 3)


Standardize Schema

In [7]:
df6 = df6.rename(columns={
    "date": "Date",
    "category": "Category",
    "amount": "Amount"
})

Fix Date Column

In [8]:
# Fix Date Column
df6["Date"] = df6["Date"].astype(str).str.strip()
df6["Date"] = pd.to_datetime(df6["Date"], errors="coerce")

# REMOVE timezone
df6["Date"] = df6["Date"].dt.tz_localize(None)

# CONVERT to date-only (YYYY-MM-DD)
df6["Date"] = df6["Date"].dt.date

In [9]:
df6.head()

,Date,Category,Amount
0,2022-07-06,Restuarant,5.50
1,2022-07-06,Market,2.00
2,2022-07-06,Coffe,30.10
3,2022-07-06,Market,17.33
4,2022-07-06,Restuarant,5.50


AMOUNT CLEANING

In [10]:
df6["Amount"] = pd.to_numeric(df6["Amount"], errors="coerce")
df6 = df6[df6["Amount"] > 0]

CATEGORY NORMALIZATION

In [11]:
df6["Category"] = df6["Category"].astype(str).str.strip().str.title()

print("\n[INFO] Unique Categories:")
print(df6["Category"].unique())


[INFO] Unique Categories:
['Restuarant' 'Market' 'Coffe' 'Transport' 'Other' 'Phone' 'Communal'
 'Clothing' 'Motel' 'Travel' 'Rent Car' 'Sport' 'Events' 'Learning'
 'Health' 'Taxi' 'Business Lunch' 'Film/Enjoyment' 'Tech' 'Joy' 'Fuel'
 'Business_Expenses']


In [12]:
df6["Category"] = df6["Category"].str.replace("_", " ", regex=False)
df6["Category"] = df6["Category"].str.title()
print("\n[INFO] Cleaned Unique Categories:")
print(df6["Category"].unique())


[INFO] Cleaned Unique Categories:
['Restuarant' 'Market' 'Coffe' 'Transport' 'Other' 'Phone' 'Communal'
 'Clothing' 'Motel' 'Travel' 'Rent Car' 'Sport' 'Events' 'Learning'
 'Health' 'Taxi' 'Business Lunch' 'Film/Enjoyment' 'Tech' 'Joy' 'Fuel'
 'Business Expenses']


CREATE TransactionID

In [13]:
df6 = df6.reset_index(drop=True)

df6["TransactionID"] = df6.index + 1
df6["TransactionID"] = df6["TransactionID"].apply(
    lambda x: f"TRX6_{str(x).zfill(6)}"
)

USER ID

In [14]:
df6["UserID"] = "US6000"

TYPE COLUMN (All Expense)

In [15]:
df6["Type"] = "Expense"

MERCHANT & ACCOUNT NAME (Empty)

In [16]:
df6["Merchant"] = ""
df6["Account Name"] = ""
df6["Currency"] = ""

In [17]:
df6.head()

,Date,Category,Amount,TransactionID,UserID,Type,Merchant,Account Name,Currency
0,2022-07-06,Restuarant,5.50,TRX6_000001,US6000,Expense,,,
1,2022-07-06,Market,2.00,TRX6_000002,US6000,Expense,,,
2,2022-07-06,Coffe,30.10,TRX6_000003,US6000,Expense,,,
3,2022-07-06,Market,17.33,TRX6_000004,US6000,Expense,,,
4,2022-07-06,Restuarant,5.50,TRX6_000005,US6000,Expense,,,


LLM — GENERATE TRANSACTION DESCRIPTION

In [18]:
from dateutil import parser
from groq import Groq

client = Groq(api_key="gsk_REDACTED")

def generate_description_groq(category, amount, txn_type):
    prompt = f"""
    Generate a short, realistic transaction description using only:
    - Category: {category}
    - Amount: {amount}
    - Type: {txn_type}

    Rules:
    - 7 to 12 words
    - Do NOT invent store names or merchants
    - Do NOT hallucinate extra details
    - Must sound like a real personal finance transaction
    - Keep it simple and human-like
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=25,
        temperature=0.4,
    )

    return response.choices[0].message.content.strip()

In [19]:
# Generate once per unique category
unique_categories = df6["Category"].unique()
desc_cache = {}

for cat in unique_categories:
    sample_row = df6[df6["Category"] == cat].iloc[0]
    txn_type = sample_row["Type"]
    amount = sample_row["Amount"]

    desc_cache[cat] = generate_description_groq(
        cat, amount, txn_type
    )

print("\n[INFO] Generated Descriptions Cache:")
print(desc_cache)

# Map to dataset
df6["Transaction Description"] = df6["Category"].map(desc_cache)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01k6ahth42ezvvsv5tkrqrfk5t` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 121. Please try again in 1m44.544s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

FINAL VALIDATION

In [ ]:
required = ["TransactionID","UserID","Date","Category","Amount","Type"]

print("\n[INFO] Missing required columns:")
print(df6[required].isna().sum())

print("\n[INFO] Final shape:", df6.shape)
print("\nTransactions per User:")
print(df6["UserID"].value_counts())

In [ ]:
df6.head()

In [ ]:
df6.tail()

SAVE CLEANED DATASET

In [ ]:
import os
os.makedirs("Tofinal", exist_ok=True)
df6.to_csv("Tofinal/budjet.csv", index=False)
print("\nFile saved successfully → Tofinal/budjet.csv")